# Video 06 - Exercise Solutions

Try the exercises in [exercises.md](../exercises.md) yourself first. These are **one** good answer each.

In [0]:
import json
from datetime import date
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"

def tool(name, description, properties):
    """Small helper to describe a tool; every property is required."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties, "required": list(properties)}}}

def run_round(question, tools, functions, model=MODEL):
    """ONE tool round that handles every tool call in the model's reply."""
    messages = [{"role": "user", "content": question}]
    msg = client.chat.completions.create(model=model, messages=messages, tools=tools).choices[0].message
    if not msg.tool_calls:
        return msg.content
    messages.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:                               # one tool message per call
        args = json.loads(call.function.arguments or "{}")
        result = functions[call.function.name](**args)
        print(f"  ran {call.function.name}({args}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
    second = client.chat.completions.create(model=model, messages=messages, tools=tools).choices[0].message
    if second.tool_calls:   # the model wants ANOTHER round - one round wasn't enough
        asked = [f"{c.function.name}({c.function.arguments})" for c in second.tool_calls]
        return f"(no final answer yet - the model asked for more tools: {asked})"
    return second.content

In [0]:
def get_weather(city: str) -> str:
    fake = {"perth": "sunny, 31C", "london": "rainy, 12C", "tokyo": "cloudy, 18C"}
    return fake.get(city.lower(), "unknown city")

WEATHER_TOOL = tool("get_weather", "Get the current weather for a city.",
                    {"city": {"type": "string", "description": "City name, e.g. Perth"}})

print(run_round("Should I bring an umbrella in London today?", [WEATHER_TOOL], {"get_weather": get_weather}))

In [0]:
# The model can ask for SEVERAL tool calls in one reply - run them all, one tool message each.
for model in [MODEL, "databricks-claude-haiku-4-5"]:
    print(f"--- {model} ---")
    print(run_round("What's the weather in Perth and in Tokyo?", [WEATHER_TOOL], {"get_weather": get_weather}, model), "\n")

# In our tests Claude asked for BOTH cities in one reply, so one round was enough.
# Llama asked for Perth first, then Tokyo in a SECOND round - so one round gave no final answer.
# Keeping going until there are no more tool calls is exactly the agent loop (Video 08).

In [0]:
TODAY_TOOL = tool("get_today", "Get today's date in ISO format (YYYY-MM-DD).", {})

print(run_round("How many days until Christmas?", [TODAY_TOOL], {"get_today": lambda: date.today().isoformat()}))

christmas = date(date.today().year, 12, 25)
if christmas < date.today():
    christmas = date(date.today().year + 1, 12, 25)
print("Python says:", (christmas - date.today()).days, "days")
# The model still does the date maths in its head - a 'days_between' tool would make it exact.

In [0]:
def buggy_multiply(a: float, b: float) -> float:
    return a * b + 1000          # BUG on purpose

MULTIPLY_TOOL = tool("multiply", "Multiply two numbers exactly. Use this for any multiplication.",
                     {"a": {"type": "number"}, "b": {"type": "number"}})

print(run_round("What is 48729 * 3817?", [MULTIPLY_TOOL], {"multiply": buggy_multiply}))
print("Correct answer:", 48729 * 3817)

# In our test both Llama and Claude repeated the wrong number confidently.
# The model trusts your tools completely: YOU are responsible for tool results being correct.